**Описание задачи**  
Добро пожаловать в 2912 год, где ваши навыки работы с данными потребуются для решения космической загадки. Мы получили сообщение с расстояния в четыре световых года, и дела обстоят не лучшим образом.

Космический корабль «Астра» — межзвездный пассажирский лайнер, запущенный месяц назад.

С почти 13 000 пассажиров на борту судно отправилось в свой первый рейс, перевозя эмигрантов из нашей Солнечной системы на три новые пригодные для жизни экзопланеты, вращающиеся вокруг ближайших звезд.
Огибая систему Альфа Центавра по пути к первому пункту назначения — знойной планете 55 Cancri E, — космический корабль «Астра» столкнулся с пространственно-временной аномалией, скрытой в пылевом облаке. Хотя корабль остался цел, почти половина пассажиров была перенесена в альтернативное измерение!

Чтобы помочь спасателям и вернуть потерянных пассажиров, вам предстоит предсказать, кого из них перенесла аномалия, используя записи, извлеченные из поврежденной компьютерной системы корабля.

Помогите спасти их и изменить историю!

**Задание**

Ваша задача — предсказать, был ли пассажир перенесен в альтернативное измерение во время столкновения космического корабля «Астра» с пространственно-временной аномалией. Для этого вам предоставлен набор личных записей, извлеченных из поврежденной компьютерной системы корабля.

Вам необходимо:

реализовать алгоритм классификации, который сможет по имеющимся данным определить, перенесен пассажир или нет;
получить предсказания на тестовых данных (файл public_test.csv);
загрузить .csv файл с предсказаниями.

In [1]:

import pandas as pd


In [2]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("public_test.csv") 

In [3]:
# Проверка загрузки данных
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nTrain columns:", train_df.columns.tolist())
print("\nTest columns:", test_df.columns.tolist())

Train shape: (8693, 14)
Test shape: (2138, 13)

Train columns: ['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Name', 'Transported']

Test columns: ['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Name']


In [20]:
import pandas as pd

# Выводим распределение целевой переменной
print("Распределение Transported:")
print(train_df['Transported'].value_counts())
print("\nВ процентном соотношении:")
print(train_df['Transported'].value_counts(normalize=True) * 100)


Распределение Transported:
Transported
True     4378
False    4315
Name: count, dtype: int64

В процентном соотношении:
Transported
True     50.362361
False    49.637639
Name: proportion, dtype: float64


In [4]:
# Проверка на пропуски
print("Пропуски в train:")
print(train_df.isnull().sum())
print("\nПропуски в test:")
print(test_df.isnull().sum())


Пропуски в train:
PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

Пропуски в test:
PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
dtype: int64


In [5]:
# Заполним пропуски в числовых столбцах средним значением
# Определим числовые столбцы (примерно, исходя из описания)
numeric_columns = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in numeric_columns:
    if col in train_df.columns:
        mean_val = train_df[col].mean()
        train_df[col] = train_df[col].fillna(mean_val)
        if col in test_df.columns: # Проверяем, существует ли столбец в тесте
            test_df[col] = test_df[col].fillna(mean_val)

In [6]:
# Заполним пропуски в категориальных и бинарных столбцах наиболее частым значением (mode)
categorical_and_binary_columns = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']
for col in categorical_and_binary_columns:
    if col in train_df.columns:
        mode_val = train_df[col].mode()[0] if not train_df[col].mode().empty else 'Unknown' # Обработка случая без моды
        train_df[col] = train_df[col].fillna(mode_val)
        if col in test_df.columns: # Проверяем, существует ли столбец в тесте
            test_df[col] = test_df[col].fillna(mode_val)

/tmp/ipykernel_3144/2379943577.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_df[col] = train_df[col].fillna(mode_val)


In [7]:
# Для столбцов Cabin и Name можно заполнить пропуски строкой 'Unknown'
if 'Cabin' in train_df.columns:
    train_df['Cabin'] = train_df['Cabin'].fillna('Unknown')
    test_df['Cabin'] = test_df['Cabin'].fillna('Unknown')

if 'Name' in train_df.columns:
    train_df['Name'] = train_df['Name'].fillna('Unknown')
    test_df['Name'] = test_df['Name'].fillna('Unknown')


In [8]:
# Создаем новую колонку с объединёнными текстовыми признаками для train_df
# Используем указанные вами колонки: HomePlanet, CryoSleep, Cabin, Destination, Name
train_df['combined_text'] = (
    train_df['HomePlanet'].astype(str) +
    ' ' + train_df['CryoSleep'].astype(str) +
    ' ' + train_df['Cabin'].astype(str) +
    ' ' + train_df['Destination'].astype(str) +
    ' ' + train_df['Name'].astype(str)
)

In [9]:
# Для test_df
test_df['combined_text'] = (
    test_df['HomePlanet'].astype(str) +
    ' ' + test_df['CryoSleep'].astype(str) +
    ' ' + test_df['Cabin'].astype(str) +
    ' ' + test_df['Destination'].astype(str) +
    ' ' + test_df['Name'].astype(str)
)


In [10]:
# Проверим результат объединения
print("train_df['combined_text'] (первые 5 строк):")
print(train_df['combined_text'].head())

print("\ntest_df['combined_text'] (первые 5 строки):")
print(test_df['combined_text'].head())

train_df['combined_text'] (первые 5 строк):
0     Europa False B/0/P TRAPPIST-1e Maham Ofracculy
1         Earth False F/0/S TRAPPIST-1e Juanna Vines
2       Europa False A/0/S TRAPPIST-1e Altark Susent
3        Europa False A/0/S TRAPPIST-1e Solam Susent
4    Earth False F/1/S TRAPPIST-1e Willy Santantines
Name: combined_text, dtype: object

test_df['combined_text'] (первые 5 строки):
0         Earth True G/73/S TRAPPIST-1e Nelly Richan
1        Mars True F/1631/S TRAPPIST-1e Bleark Weeke
2     Earth False F/885/P 55 Cancri e Courta Johnsby
3    Europa False A/31/S TRAPPIST-1e Jabbab Entenedy
4     Earth True G/958/P TRAPPIST-1e Blancy Moongton
Name: combined_text, dtype: object


In [16]:
train_df.columns

Index(['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age',
       'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'Name', 'Transported', 'combined_text'],
      dtype='object')

In [15]:
train_df

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,combined_text
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False,Europa False B/0/P TRAPPIST-1e Maham Ofracculy
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True,Earth False F/0/S TRAPPIST-1e Juanna Vines
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False,Europa False A/0/S TRAPPIST-1e Altark Susent
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False,Europa False A/0/S TRAPPIST-1e Solam Susent
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True,Earth False F/1/S TRAPPIST-1e Willy Santantines
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,9276_01,Europa,False,A/98/P,55 Cancri e,41.0,True,0.0,6819.0,0.0,1643.0,74.0,Gravior Noxnuther,False,Europa False A/98/P 55 Cancri e Gravior Noxnuther
8689,9278_01,Earth,True,G/1499/S,PSO J318.5-22,18.0,False,0.0,0.0,0.0,0.0,0.0,Kurta Mondalley,False,Earth True G/1499/S PSO J318.5-22 Kurta Mondalley
8690,9279_01,Earth,False,G/1500/S,TRAPPIST-1e,26.0,False,0.0,0.0,1872.0,1.0,0.0,Fayey Connon,True,Earth False G/1500/S TRAPPIST-1e Fayey Connon
8691,9280_01,Europa,False,E/608/S,55 Cancri e,32.0,False,0.0,1049.0,0.0,353.0,3235.0,Celeon Hontichre,False,Europa False E/608/S 55 Cancri e Celeon Hontichre


In [17]:
# Пример списка числовых колонок (подставьте свои)
num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import OneHotEncoder


# Определяем препроцессор для текстовых данных
text_transformer = Pipeline([
    ('tfidf', TfidfVectorizer())
])


# Препроцессор для числовых данных
numeric_transformer = Pipeline([
    ('scaler', StandardScaler())
])


# Объединяем оба трансформера
preprocessor = ColumnTransformer(
    transformers=[
        ('text', text_transformer, 'combined_text'),
        ('numeric', numeric_transformer, num_cols)
    ])

# Полная модель
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [19]:
# Определяем признаки (X) и целевую переменную (y)
X = train_df[['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age',
       'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'Name', 'combined_text']]

y = train_df['Transported']  # целевая переменная

# Разделяем данные на обучающую и валидационную выборки
X_train, X_val, y_train, y_val = train_test_split(
    X, 
    y,
    test_size=0.2,           # 20% данных для валидации
    random_state=42,      # воспроизводимость разбиения
    stratify=y            # сохранение баланса классов в выборках
)


In [21]:
# Создаем полную модель
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Обучение модели
model.fit(X_train, y_train)

# Прогнозирование и расчет F1-метрики
val_preds = model.predict(X_val)
print(f'F1-score на валидации: {f1_score(y_val, val_preds, average="macro")}')

F1-score на валидации: 0.7737387026940055


In [22]:
# Простая модель для теста
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Обучение модели
model.fit(X_train, y_train)

# Прогнозирование и расчёт F1-метрики
val_preds = model.predict(X_val)
print(f'F1-score на валидации: {f1_score(y_val, val_preds, average="macro")}')

F1-score на валидации: 0.7928331466965286


In [23]:
from sklearn.metrics import precision_recall_curve, average_precision_score, classification_report, f1_score

# Предсказание вероятностей для PR-AUC
y_val_proba = model.predict_proba(X_val)[:, 1]

# 1. PR-AUC
pr_auc = average_precision_score(y_val, y_val_proba)
print(f'PR-AUC на валидации: {pr_auc:.4f}')

# 2. F1-score (макро)
val_preds = model.predict(X_val)
f1_macro = f1_score(y_val, val_preds, average='macro')
print(f'F1-score (макро) на валидации: {f1_macro:.4f}')

# 3. Classification Report
print("\nClassification Report:")
print(classification_report(y_val, val_preds, target_names=['Не перенесен', 'перенесен']))

PR-AUC на валидации: 0.8832
F1-score (макро) на валидации: 0.7928

Classification Report:
              precision    recall  f1-score   support

Не перенесен       0.80      0.77      0.79       863
   перенесен       0.78      0.81      0.80       876

    accuracy                           0.79      1739
   macro avg       0.79      0.79      0.79      1739
weighted avg       0.79      0.79      0.79      1739



In [24]:
model.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('text', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [25]:
x = test_df[['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age',
       'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'Name', 'combined_text']]

In [27]:
test_df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,combined_text
0,0495_01,Earth,True,G/73/S,TRAPPIST-1e,4.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Richan,Earth True G/73/S TRAPPIST-1e Nelly Richan
1,8464_02,Mars,True,F/1631/S,TRAPPIST-1e,34.0,False,0.0,0.0,0.0,0.0,0.0,Bleark Weeke,Mars True F/1631/S TRAPPIST-1e Bleark Weeke
2,4277_01,Earth,False,F/885/P,55 Cancri e,43.0,False,0.0,737.0,0.0,0.0,4.0,Courta Johnsby,Earth False F/885/P 55 Cancri e Courta Johnsby
3,2380_01,Europa,False,A/31/S,TRAPPIST-1e,32.0,False,0.0,3561.0,45.0,1552.0,7161.0,Jabbab Entenedy,Europa False A/31/S TRAPPIST-1e Jabbab Entenedy
4,5918_01,Earth,True,G/958/P,TRAPPIST-1e,46.0,False,0.0,0.0,0.0,0.0,0.0,Blancy Moongton,Earth True G/958/P TRAPPIST-1e Blancy Moongton


In [26]:
# Прогнозирование и оценка модели
y_pred = model.predict(x)

In [28]:
submission_df = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Transported': y_pred
})

In [29]:
# Просмотрим первые строки получившегося DataFrame:
print(submission_df.head())

  PassengerId  Transported
0     0495_01         True
1     8464_02         True
2     4277_01         True
3     2380_01        False
4     5918_01         True


In [30]:
submission_df.to_csv('t_r_submission.csv', index=False)